# Design-Aufgabe: Regelung eines mobilen Roboters

In dieser großen Übungsaufgabe setzt du die **Objektorientierung** (Arbeitsblatt 07) auf einer realistischeren
Struktur um: ein **mobiler Roboter**, der mit **Sensoren** seine Umgebung erfasst und mit **Motoren** darauf reagiert.

Du erstellst eine saubere **Klassenhierarchie** (abstrakte Basisklassen, Vererbung, Polymorphie) und nutzt sie in einem
**vorgegebenen Regelungs-Hauptprogramm** (ohne das Hauptprogramm zu verändern).

## Lernziele

- Eine abstrakte Basisklasse mit `abc` als **Schnittstellen-Vertrag** entwerfen.
- Eine zusammenhängende **Vererbungshierarchie** erstellen (3 Sensorarten, 2 Motorarten).
- **Polymorphie** nutzen: Das Hauptprogramm spricht nur gegen die Basisklassen-Schnittstelle.
- Erstes **System-Design**: Schnittstellen festlegen, dann implementieren.

> **Aufgabenstellung:** Das Hauptprogramm unten ist **vorgegeben und darf nicht geändert werden**.
> Deine Aufgabe ist es, die Klassen `Ultraschallsensor`, `Lichtsensor`, `Gyrosensor` (aus `Sensor`)
> sowie `Gleichstrommotor`, `Schrittmotor` (aus `Motor`) so zu implementieren, dass das Programm fehlerfrei läuft.

## Teil 1 – Vorgegebenes Hauptprogramm

Die Zelle definiert eine Funktion `main()`, die später ausgeführt wird. Sie hat folgende Funktion:

1. nimmt zwei `dict` mit **Sensoren** und **Motoren** entgegen,
2. ruft in jedem Regelzyklus den Regelagorithmus `roboter_steuerung` auf,
3. **bremst**, wenn ein Hindernis zu nah ist, sonst **fährt** es (Geschwindigkeit an Helligkeit gekoppelt),
4. zeichnet in einer **Live-Anzeige** kontinuierlich Sensorbalken, Zustand und Motordrehzahlen (kleiner Trick mit Jupyter-spezifischen Funktionen).

Es muss mindestens ein "abstand"- und ein "licht"-Sensor vorhanden sein.

> **Vertrag (Schnittstelle), die deine Klassen erfüllen müssen:**
>
> Jeder Sensor bietet `name` (Property), `messen() -> float` und `info() -> str`.
>
> Jeder Motor bietet `name` (Property), `drehzahl_setzen(drehzahl)`, `aktuelle_drehzahl() -> float`,
> `bremsen()` und `info() -> str`.

In [ ]:
# Animation und Ausgabe sind in einer externen Datei
# Sie sind vorgegeben und nicht Teil der Übung
from simulation import text_animation

def roboter_steuerung(sensoren, motoren, zyklen=12, abstand_grenzwert=40.0):
    """Live-Regelungs-Simulation: Sensoren lesen, Motoren steuern, Aktion anzeigen."""
    abstand = sensoren["abstand"].messen()
    licht   = sensoren["licht"].messen()

    if abstand < abstand_grenzwert:
        for m in motoren.values():
            m.bremsen()
        zustand = "ANHALTEN  (Hindernis!)"
    else:
        drehzahl = 15.0 + 0.06 * licht        # einfacher P-Regler
        for m in motoren.values():
            m.drehzahl_setzen(drehzahl)
        zustand = "FAHREN"

    return zustand


def main(sensoren, motoren, zyklen=12):
    if "abstand" not in sensoren:
        raise ValueError("Der Roboter braucht einen Abstandssensor")
    if "licht" not in sensoren:
        raise ValueError("Der Roboter braucht einen Licht-Sensor")

    for zyklus in range(1, zyklen + 1):
        zustand = roboter_steuerung(sensoren, motoren)
        text_animation(zyklus, zyklen, zustand, sensoren, motoren)
    print("\nFertig – Regelsimulation beendet.")


    for s in sensoren.values():
        print("  ", s.info())
    for m in motoren.values():
        print("  ", m.info())


## Teil 2 – Aufgabe 1: Sensor-Hierarchie (3 Sensorarten)

Entwirf eine abstrakte Basisklasse `Sensor` (aus `abc`) mit:

- Konstruktor, der einen Parameter für den Namen bekommt und das Attribut `_name` setzt,
- einer **abstrakten Methode** `messen(self) -> float`,
- einer konkreten Methode `info(self) -> str`.

Leite daraus **drei** konkrete Sensoren ab, die `messen()` jeweils mit simulierten Messwerten implementieren. Du kannst z.B. `import random` machen und mit `random.uniform(min, max)` Zufallszahlen generieren. Du darfst die Messwerte aber auch anders erzeugen.

| Klasse | Messgröße | Messwert-Bereich (Einheit) |
|---|---|---|
| `Ultraschallsensor` | Abstand | 5 … 300 (cm); Konstruktor nimmt optional `max_reichweite` |
| `Lichtsensor` | Helligkeit | 0 … 1000 (Lux) |
| `Gyrosensor` | Winkelgeschwindigkeit | −90 … +90 (Grad/s) |


In [ ]:
import random
from abc import ABC, abstractmethod

# Hier dein Code

## Teil 3 – Aufgabe 2: Motor-Hierarchie (2 Motorarten)

Entwirf eine abstrakte Basisklasse `Motor` (aus `abc`) mit:

- Konstruktor, der `_name` und die aktuelle Drehzahl `_drehzahl = 0.0` setzt,
- einer **abstrakten Methode** `drehzahl_setzen(drehzahl) -> None`,
- einer konkreten Methode `aktuelle_drehzahl() -> float` und `bremsen()` (setzt Drehzahl auf 0),
- einer konkreten Methode `info() -> str`.

Leite daraus **zwei** konkrete Motoren ab, die `drehzahl_setzen()` unterschiedlich umsetzen:

| Klasse | Eigenschaft |
|---|---|
| `Gleichstrommotor` | **stufenlose** (kontinuierliche) Drehzahl, Werte auf ±100 begrenzt |
| `Schrittmotor` | **ganzzahlige** Drehzahl (voll diskret), auf ±100 begrenzt; Konstruktor nimmt optional `schritte_pro_umdrehung` |

Beide Motorarten zeigen damit über **dieselbe Schnittstelle** unterschiedliches Verhalten – genau das ist Polymorphie.

In [ ]:
# Hier dein Code

## Teil 4 – Ausführen

Jetzt sind alle Klassen definiert. Instanziiere die Objekte und führe `main()` aus – es sollte eine Live-Simulation über 12 Schritte ohne Fehler ausgeben.

In [ ]:
# Passe den Code auf deine Klassennamen an.
sensoren = {
    "abstand": Ultraschallsensor("Ultraschall vorn"),
    "licht":   Lichtsensor("Lichtsensor"),
    "gyro":    Gyrosensor("Gyroskop"),
}
motoren = {
    "links":  Gleichstrommotor("DC-Motor links"),
    "rechts": Schrittmotor("Schrittmotor rechts"),
}

# Führe diese Zelle aus, um die Live-Regelsimulation zu starten:
main(sensoren, motoren)

## Teil 4b – grafische Visualisierung

Zusätzlich zur Text-Live-Anzeige kannst du die Regelung **als animierte Draufsicht** darstellen.
Der Roboter ist ein Kreis, der **Ultraschall-Kegel** zeigt die gemessene Reichweite (rot = Hindernis bremst),
und die Motordrehzahlen werden als Text eingeblendet.

Diese Anzeige dient nur zur Unterhaltung. Der Informationsgehalt ist der selbe wie bei der Simulation oben, es sieht nur hübscher aus. Die technischen Details von grafischen Darstellungen lernen wir in **zukünftigen Vorlesungen** kennen.

Es kann sein, dass diese Simulation in deiner konkreten Jupyter-Umgebung *nicht* funktioniert. Darum gibt es oben die Text-Variante, die sollte in jeder Jupyter-Umgebung laufen.

In [ ]:
from simulation import matplotlib_animation

await matplotlib_animation(sensoren, motoren, roboter_steuerung)

## Teil 5 – Aufgabe 3: Erweitern ohne das Hauptprogramm anzufassen

Ein wichtiger Vorteil der geschlossenen Klassenhierarchie: **neue Sensor- oder Motorarten lassen sich
nachrüsten, ohne das Hauptprogramm zu verändern**.

Implementiere einen **vierten Sensor** `Kompasssensor`, der einen simulierten Kurs (0 … 360 Grad) misst,
und stelle ihn in `main()` in das Wörterbuch der Sensoren – mehr ist nicht nötig.

**Zusatz-Frage:** Warum muss `roboter_steuerung` dabei nicht geändert werden?

In [ ]:
# Hier deine Klasse:
...

# Kompass in die Sensor-Tabelle einbauen (main() ist unveraendert)
sensoren_mit_kompass = {
    "abstand": Ultraschallsensor("Ultraschall vorn"),
    "licht":   Lichtsensor("Lichtsensor"),
    "gyro":    Gyrosensor("Gyroskop"),
    "kompass": ... # hier dein Objekt
}

main(sensoren_mit_kompass, motoren)


## Teil 6 – Bonus-Aufgabe (optional): Schnittstellen

Zuletzt wollen wir noch die Vorteile von Schnittstellen testen: Definieren von gleichartigen Aufrufen, ohne dass dafür eine Vererbungs-Hierarchie benötigt wird.

1. Erstelle eine Schnittstellen-Klasse `Nothaltend` mit einer Methode `nothalt()`.
2. Füge diese Schnittstelle zu allen deinen Klassen hinzu. Objekte sollen nach einem Nothalt keinerlei Bewegung mehr durchführen und keine Messungen zurückliefern (immer Wert 0).
3. Ändere `roboter_regelung` so ab, dass bei Unterschreiten eines Abstandes von 10cm ein Nothalt aller Sensoren und Aktoren durchgeführt wird.

Es lohnt sich, diese Änderungen in einer Kopie dieses Arbeitsblattes zu machen.

## Zusammenfassung

- Eine abstrakte Basisklasse (`abc`) legt die **Schnittstelle** fest; konkrete Unterklassen implementieren sie.
- Das Hauptprogramm ist **gegen die Basisklasse geschrieben** → es funktioniert mit jeder Unterklasse (**Polymorphie**, späte Bindung).
- Unterschiedliche Sensoren/Motoren zeigen über dieselbe Schnittstelle **unterschiedliches Verhalten**.
- Neue Klassen lassen sich **nachrüsten, ohne bestehenden Code zu ändern**.
